# Research: Earnings Volatility Ramp

## Objective
Obtain the universe for all the earnings dates for liquid stocks (with options). 
**NOTE:** Switched to 2023-2024 window for testing due to missing 2010 data locally. Revert dates if historical data is installed.

## Steps
1. Initialize QB with start/end dates (2023-2024).
2. Select initial liquid universe (Top 500 by Dollar Volume) using Universe History sampling.
3. Fetch earnings dates for these symbols.
4. Filter for symbols that have options available at the time of earnings.
5. Output the result.

In [ ]:
from AlgorithmImports import *
import pandas as pd

# 1. Initialize QuantBook
qb = QuantBook()

# 2. Set Time Period (2023-2024)
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 1, 1)
qb.SetStartDate(start_date)
qb.SetEndDate(end_date)

print(f"Researching period: {start_date.date()} to {end_date.date()}")

# DIAGNOSTIC: Check if we have ANY data for this period.
print("DIAGNOSTIC: Checking for SPY data in 2023-01...")
spy = qb.AddEquity("SPY", Resolution.Daily)
h_spy = qb.History(qb.Securities.Keys, start_date, start_date + timedelta(days=10))

if h_spy.empty:
    print("CRITICAL WARNING: No history found for SPY in Jan 2023.")
else:
    print(f"Success: Found {len(h_spy)} rows of SPY data. Price data exists.")


Step 1: Broad Universe Selection (Liquidity Proxy)

In [ ]:
print("Selecting initial liquid universe using Quarterly Sampling...")

# Ensure resolution is Daily for Coarse data
qb.UniverseSettings.Resolution = Resolution.Daily

# We use qb.UniverseHistory to get the coarse data.
# We must add a universe first to query its history.
def coarse_selector(coarse):
    return [c.Symbol for c in coarse if c.DollarVolume > 10_000_000]

u = qb.AddUniverse(coarse_selector)

unique_symbols = set()

# Sample quarterly to capture stocks that might become liquid or IPO
sample_dates = []
curr = start_date
while curr < end_date:
    sample_dates.append(curr)
    curr = curr + timedelta(days=90)

for d in sample_dates:
    # Fetch a small window around the date to ensure we hit a trading day
    # UniverseHistory returns the coarse data passed into the filter.
    try:
        history = qb.UniverseHistory(u, d, d + timedelta(days=5))
        
        data_found = False
        for collection in history:
            if not collection.Data:
                continue
                
            # Debug: print count of raw items
            # print(f"Raw items on {collection.Time}: {len(collection.Data)}")
                
            for c in collection.Data:
                if c.DollarVolume > 10_000_000 and c.Price > 10:
                    unique_symbols.add(c.Symbol)
                    data_found = True
            
            if data_found:
                print(f"Sampled {d.date()} -> Found data. Unique Count: {len(unique_symbols)}")
                break # Only need one day per quarter
        
        if not data_found:
             print(f"Sampled {d.date()} -> No liquid stocks found in window (Missing Coarse Data?)")

    except Exception as e:
        print(f"Error sampling {d.date()}: {e}")
        pass

candidate_symbols = list(unique_symbols)
print(f"Final Candidate Count: {len(candidate_symbols)}")


Step 2: Get Earnings Dates for these Symbols

In [ ]:
print("Fetching earnings dates...")

# Chunking to be safe
chunk_size = 500
earnings_events = []

if not candidate_symbols:
    print("Skipping earnings fetch (No candidates found).")
else:
    for i in range(0, len(candidate_symbols), chunk_size):
        chunk = candidate_symbols[i:i+chunk_size]
        print(f"Processing chunk {i} to {i+len(chunk)}...")
        
        try:
            earnings_data = qb.GetFundamental(chunk, "Earnings.AnnouncementDate", start_date, end_date)
            
            if not earnings_data.empty:
                for symbol in earnings_data.columns:
                    series = earnings_data[symbol].dropna()
                    for timestamp, announce_date in series.items():
                        try:
                            if isinstance(announce_date, pd.Timestamp):
                                e_date = announce_date.date()
                            else:
                                e_date = pd.to_datetime(str(announce_date)).date()
                            
                            if start_date.date() <= e_date <= end_date.date():
                                earnings_events.append({'Symbol': symbol, 'EarningsDate': e_date})
                        except:
                            pass
        except Exception as e:
            print(f"Chunk error: {e}")

    print(f"Found {len(earnings_events)} earnings events.")


Step 3: Filter for Option Availability

In [ ]:
print("Checking for option availability...")

final_universe = []

if not earnings_events:
    print("Skipping option check (No earnings events found).")
else:
    for idx, event in enumerate(earnings_events):
        if idx % 200 == 0 and idx > 0: 
            print(f"Checked {idx} events...")
            
        sym = event['Symbol']
        e_date = event['EarningsDate']
        check_dt = datetime(e_date.year, e_date.month, e_date.day)
        
        try:
            contracts = qb.OptionChainProvider.GetOptionContractList(sym, check_dt)
            if contracts:
                event['HasOptions'] = True
                event['ContractCount'] = len(contracts)
                final_universe.append(event)
        except:
            pass

Step 4: Output Results

In [ ]:
df_results = pd.DataFrame(final_universe)
if not df_results.empty:
    df_results['Symbol'] = df_results['Symbol'].apply(lambda s: s.Value)
    df_results = df_results.sort_values(by=['EarningsDate', 'Symbol'])

print(f"Final Universe Count: {len(df_results)}")
if not df_results.empty:
    print(df_results.head(20))